# 4장 실습
---

In [8]:
import platform
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from matplotlib import rc

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

print(f"현재 사용 중인 연산 디바이스 : {device}")

# matplotlib 한글 깨짐 방지 설정 
if platform.system() == 'Darwin':      # macOS 환경
    rc('font', family='AppleGothic')
elif platform.system() == 'Windows':   # Windows 환경
    rc('font', family='Malgun Gothic')
    
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

현재 사용 중인 연산 디바이스 : cuda


In [9]:
# Fashion Minist 28 x 28(2차원) (0~255) -> 표준 점수 -> 특성은 1차원으로 변환
# 60,000 -> 훈련 데이터는 50,000장, 검증(10,000)

# 훈련 데이터 셋
raw_train_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=ToTensor()
)

# 테스트 데이터 셋
test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

In [10]:
#검증 데이터 셋 분리
train_size = 50000
val_size = 10000

train_data, val_data = random_split(
    raw_train_data, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"데이터 분할 완료 -> 훈련용: {len(train_data)}장 | 검증용: {len(val_data)}장 | 테스트용: {len(test_data)}장")

데이터 분할 완료 -> 훈련용: 50000장 | 검증용: 10000장 | 테스트용: 10000장


In [19]:
import torch.nn as nn
import torch.optim as optim

class AdvancedFashiorClassifier(nn.Module):

    # 사용할 층(layer)를 정의
    def __init__(self):
        super(AdvancedFashiorClassifier, self).__init__()
        # (28, 28) -> 특성은 1차원으로 변경 (784,)
        self.flatten = nn.Flatten()
        # 입력층(784) -> 은닉층(64) 선형 변환
        self.linear1 = nn.Linear(28 * 28, 256)
        # 활성화 함수로 은닉층에 비선형성 주입을 위한 ReLU 지정
        self.relu = nn.ReLU()
        # 학습 중 뉴런의 20%를 무작위로 비활성화하는 드롭아웃 설정
        self.dropout = nn.Dropout(p=0.5)
        # 은닉층(64) -> 출력층(10) 선형 변환
        self.linear2 = nn.Linear(256, 10)
    
    def __str__(self):
        return f"{super().__str__()}, "

    def __repr__(self):
        return f"{super().__repr__()}, "

    def forward(self, x):
        out = self.flatten(x)
        out = self.linear1(out)
        out = self.relu(out)
        out = self.dropout(out) # 훈련 시에만 적용됨, 과대적합 방지
        out = self.linear2(out) # 별도의 Softmax 없이 Raw 점수(Logits) 출력
        return out


In [20]:
model = AdvancedFashiorClassifier().to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: {total_params:,}개")

모델의 가속기 이관 완료 | 학습 가능한 총 파라미터 수: 203,530개


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [15]:
# 데이터 세트별 DataLoader 생성
# batch_size -> 에포크에서 전 데이터 셋을 32개의 배치로 분할 -> 미니배치 경사하강법
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)   # 데이터 섞기 활성화 (과적합 방지)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)     # 평가는 섞을 필요 없음
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)   # 평가는 섞을 필요 없음

In [18]:
epochs = 5 # 최대 에포크
patience = 3 # 검증 로스가 더이상 떨어지지 않는 횟수, 3회 이상이면 학습 종료(조기 종료)
patience_count = 0
best_val_loss = float("inf")

train_losses = []
val_losses = []

print("\n=== 모델 학습 및 검증을 시작합니다. ===")
for epoch in range(epochs):
    # --- [훈련 모드]---
    running_loss = 0
    model.train() # 드롭아웃 등 훈련 전용 기능 활성화

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() # 이전 그래디언트(기울기)를 초기화

        # 순방향 예측
        outputs = model(images)

        # 손실 계산
        loss = criterion(outputs, labels)

        # 오차 역전파 수행 - 그래디언트(기울기)
        loss.backward()

        # 옵티마이저를 통한 가중치 업데이트
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader) # 한 에포크당 평균 로스(배치 로스의 평균)
    print(f"{epoch + 1}번째 평균 훈련 손실값 : {epoch_loss:.4f}")


=== 모델 학습 및 검증을 시작합니다. ===
1번째 평균 훈련 손실값 : 0.4419
2번째 평균 훈련 손실값 : 0.4279
3번째 평균 훈련 손실값 : 0.4281
4번째 평균 훈련 손실값 : 0.4167
5번째 평균 훈련 손실값 : 0.4229


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

df = pd.read_csv("ratings.txt", sep='\t')
#df.info()

df = df.dropna()

train_tmp_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df, val_df = train_test_split(train_tmp_df, test_size=0.2, random_state=42)

vocab_size = 20000
all_words = []
for text in train_df['document'].astype(str):
    all_words.extend(text.split())

counts = Counter(all_words)
vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(vocab_size - 2))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
top_3_word_counts = counts.most_common(3)
for i in range(3):
    print(f"빈도수 {i + 1}위 단어: ", top_3_word_counts[i][0])
    print("나온 횟수 : ", top_3_word_counts[i][1])
    print("고유 번호 : ", vocab[top_3_word_counts[i][0]])

빈도수 1위 단어:  영화
나온 횟수 :  9296
고유 번호 :  2
빈도수 2위 단어:  너무
나온 횟수 :  7052
고유 번호 :  3
빈도수 3위 단어:  정말
나온 횟수 :  6616
고유 번호 :  4


In [54]:
class NSMCDataset(Dataset):
    def __init__(self, df, vocab, max_len=32):
        self.labels = df['label'].values
        self.texts = df['document'].astype(str).values
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # 문장을 단어로 쪼개고 숫자 인덱스로 변환 (사전에 없으면 <UNK>인 1로 매핑)
        tokens = text.split()
        ids = [self.vocab.get(token, 1) for token in tokens[:self.max_len]]
        
        # 문장 표준 규격을 위한 고정 크기 패딩(Padding) 처리
        if len(ids) < self.max_len:
            ids += [0] * (self.max_len - len(ids))
            
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

train_dataset = NSMCDataset(train_df, vocab)
val_dataset = NSMCDataset(val_df, vocab)
test_dataset = NSMCDataset(test_df, vocab)

tensor ,label = train_dataset[0]

print(len(tensor))
print(f"tensor = {tensor}\n label = {label}")

32
tensor = tensor([17098,     1,     1,     1,     1,     1,  4257,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])
 label = 1


In [ ]:
embedding_layer = nn.Embedding(num_embeddings = 20000, embedding_dim = 128)

tensor

tensor(1)